# Phase 4 · Architecture 2: LegalSeq2Seq

In [13]:
from pathlib import Path
import shutil

SOURCE = Path('/kaggle/input/datasets/abdakt/p4-a2-resume/phase4_a2_scratch_train_only')
OUT = Path('/kaggle/working/phase4_a4')

shutil.copytree(SOURCE, OUT, dirs_exist_ok=True)

PosixPath('/kaggle/working/phase4_a4')

In [2]:
%pip -q install "tokenizers==0.21.4" "huggingface_hub==0.34.4" "cloudpickle==3.1.1" scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, classification_report

BASE = Path('/kaggle/input/datasets/abdakt/a2-p3')
HF_TOKEN = os.environ['HF_TOKEN']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, stride=384,
         chunks=6, chunk_batch=2, micro=1, accum=8, rank=16,
         eval_steps=200, patience=5, classes=2)
METHODS = {
    'full': dict(epochs=6, lr=2e-5, coverage='uniform'),
    'partial': dict(epochs=8, lr=5e-5, coverage='head_tail'),
    'lora': dict(epochs=10, lr=2e-4, coverage='uniform'),
    'lorma': dict(epochs=10, lr=2e-4, coverage='uniform'),
    'loha': dict(epochs=10, lr=2e-4, coverage='all'),
    'distill': dict(epochs=8, lr=4e-5, coverage='uniform'),
}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])


## Training and development data

In [22]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = {'train':'Train/train.json', 'dev':'Train/dev.json'}
tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
R['pad'], R['eos'] = tok.token_to_id('<pad>'), tok.token_to_id('<eos>')
R['bos'] = tok.token_to_id('<bos>')

def read_docs(raw, labelled):
    if isinstance(raw, dict):
        raw = [raw] if 'id' in raw and 'text' in raw else [dict(v, id=k) for k,v in raw.items()]
    docs = []
    for row in raw:
        text = row['text']
        if isinstance(text, list):
            text = '\n'.join(text)
        y = -100
        if labelled:
            y = row['label'] if 'label' in row else row['labels']
            if isinstance(y, list) and len(y)==1: y = y[0]
            if y not in [0,1,'0','1']:
                raise ValueError('Expected a binary label for '+str(row['id']))
            y = int(y)
        docs.append(dict(id=row['id'], text=text, y=y))
    return docs

def cache_docs(split, docs):
    rows, offset = [], 0
    with (OUT/f'{split}_tokens.bin').open('wb') as f:
        for doc in docs:
            ids = np.asarray(tok.encode(doc['text'],add_special_tokens=False).ids, dtype=np.int32)
            ids.tofile(f)
            row = dict(id=doc['id'], offset=offset, size=len(ids))
            if split!='test': row['y'] = doc['y']
            rows.append(row)
            offset += len(ids)
        if offset==0: np.array([R['eos']],dtype=np.int32).tofile(f)
    print(f'{split}: {len(rows):,} cases | {offset:,} BPE tokens')
    return rows

DATA = {}
for split,name in files.items():
    path = hf_hub_download(REPO, 'Phase_4_CJPE/'+name, repo_type='dataset',
                           revision=revision, token=HF_TOKEN)
    docs = read_docs(json.loads(Path(path).read_text()), labelled=True)
    DATA[split] = cache_docs(split,docs)
    del docs
with (OUT/'train_dev.pkl').open('wb') as f: cloudpickle.dump(DATA,f)
identity = dict(config=C, settings=R.copy(), methods=METHODS, revision=revision,
                data_policy='train_only_separate_inference_v2',
                adapter_implementation='handwritten_pytorch_v1', architecture=2,
                tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest())
with (BASE/'best.pt').open('rb') as f:
    digest = hashlib.sha256()
    for block in iter(lambda:f.read(8*1024*1024),b''): digest.update(block)
identity['base_sha256'] = digest.hexdigest()
R['run_id'] = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
manifest = dict(**identity, run_id=R['run_id'], repo=REPO, files=files)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'tokenizer.json',OUT/'tokenizer.json')


train: 37,387 cases | 149,839,244 BPE tokens
dev: 1,988 cases | 10,043,322 BPE tokens


PosixPath('/kaggle/working/phase4_a4/tokenizer.json')

## LegalSeq2Seq encoder–decoder

In [23]:
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-5)
        return z.to(x.dtype) * self.weight


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        self.heads = cfg['heads']
        self.head_dim = cfg['dim'] // self.heads
        self.causal = causal
        self.qkv = nn.Linear(cfg['dim'], 3 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        inv = 10000 ** (-torch.arange(0, self.head_dim, 2).float() / self.head_dim)
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos, sin = phase.cos()[None, None].to(x.dtype), phase.sin()[None, None].to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        old = 0 if past is None else past[0].size(2)
        positions = torch.arange(old, old + t, device=x.device)
        q, k = self.rope(q, positions), self.rope(k, positions)
        if past is not None:
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None if mask is None else mask[:, None, None, :]
        fast_causal = self.causal and past is None and mask is None
        if self.causal and not fast_causal:
            causal_mask = torch.arange(k.size(2), device=x.device)[None, :] <= positions[:, None]
            allowed = causal_mask[None, None] if allowed is None else allowed & causal_mask[None, None]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          is_causal=fast_causal, dropout_p=0.0)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.head_dim = cfg['heads'], cfg['dim'] // cfg['heads']
        self.q = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        self.kv = nn.Linear(cfg['dim'], 2 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)

    def project_memory(self, memory):
        b, s, _ = memory.shape
        k, v = self.kv(memory).view(b, s, 2, self.heads, self.head_dim).unbind(2)
        return k.transpose(1, 2), v.transpose(1, 2)

    def forward(self, x, memory, mask, cached=None):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)
        k, v = self.project_memory(memory) if cached is None else cached
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :], dropout_p=0.0)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, d))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn = FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)[0]
        return x + self.ffn(self.norm2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = (RMSNorm(cfg['dim']) for _ in range(3))
        self.attn = SelfAttention(cfg, causal=True)
        self.cross = CrossAttention(cfg)
        self.ffn = FeedForward(cfg)

    def forward(self, x, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), target_mask, past, use_cache)
        x = x + a
        x = x + self.cross(self.norm2(x), memory, source_mask, cross_kv)
        return x + self.ffn(self.norm3(x)), cache


class LegalSeq2Seq(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg['encoder_layers'])])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg['decoder_layers'])])
        self.encoder_norm, self.decoder_norm = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.encoder:
            for weight in (block.attn.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(2 * len(self.encoder)))
        for block in self.decoder:
            for weight in (block.attn.out.weight, block.cross.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(3 * len(self.decoder)))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask):
        x = self.embed(source)
        for block in self.encoder:
            if self.training and self.cfg['gradient_checkpointing']:
                x = checkpoint(block, x, source_mask, use_reentrant=False)
            else:
                x = block(x, source_mask)
        return self.encoder_norm(x)

    def decode(self, target, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        x, caches = self.embed(target), []
        for i, block in enumerate(self.decoder):
            if self.training and self.cfg['gradient_checkpointing']:
                def layer(z, mem, module=block):
                    return module(z, mem, source_mask, target_mask)[0]
                x = checkpoint(layer, x, memory, use_reentrant=False)
            else:
                x, cache = block(x, memory, source_mask, target_mask,
                                 None if past is None else past[i],
                                 None if cross_kv is None else cross_kv[i], use_cache)
                caches.append(cache)
        return self.decoder_norm(x), caches

    def forward(self, source, target, labels=None):
        source_mask = source.ne(self.cfg['pad_id'])
        memory = self.encode(source, source_mask)
        x, _ = self.decode(target, memory, source_mask, target.ne(self.cfg['pad_id']))
        if labels is None:
            return F.linear(x, self.embed.weight)
        h, y = x.flatten(0, 1), labels.flatten()
        total = h.new_zeros((), dtype=torch.float32)
        for start in range(0, len(y), 256):
            def part(z, truth):
                return F.cross_entropy(F.linear(z, self.embed.weight).float(), truth,
                                       ignore_index=-100, reduction='sum')
            if self.training:
                total = total + checkpoint(part, h[start:start+256], y[start:start+256], use_reentrant=False)
            else:
                total = total + part(h[start:start+256], y[start:start+256])
        return total


In [35]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32, dropout=.05):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank,base.in_features))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.scale, self.drop = alpha/rank, nn.Dropout(dropout)
    def forward(self, x):
        update = F.linear(F.linear(self.drop(x),self.a),self.b)
        return self.base(x) + self.scale*update

class LoRMALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32, dropout=.05):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank,base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.scale, self.drop = alpha/rank, nn.Dropout(dropout)
    def forward(self, x):
        y = F.linear(x,self.base.weight)
        update = F.linear(F.linear(self.drop(y),self.a),self.b)
        y = y + self.scale*update
        return y if self.base.bias is None else y+self.base.bias

class LoHaLinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=.05):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.a1 = nn.Parameter(torch.empty(rank,base.in_features))
        self.a2 = nn.Parameter(torch.empty(rank,base.in_features))
        self.b1 = nn.Parameter(torch.zeros(base.out_features,rank))
        self.b2 = nn.Parameter(torch.empty(base.out_features,rank))
        for p in [self.a1,self.a2,self.b2]: nn.init.kaiming_uniform_(p,a=math.sqrt(5))
        self.scale, self.drop = alpha/rank, nn.Dropout(dropout)
    def forward(self, x):
        delta = (self.b1@self.a1)*(self.b2@self.a2)
        return self.base(x)+self.scale*F.linear(self.drop(x),delta)

class CaseModel(nn.Module):
    def __init__(self, lm, method, r, coverage):
        super().__init__()
        self.lm, self.r, self.coverage = lm, r, coverage
        d = lm.cfg['dim']
        self.chunk_head = nn.Sequential(nn.Linear(2*d+1,256),nn.GELU(),nn.Dropout(.15))
        self.pool = nn.Linear(256,1)
        self.head = nn.Sequential(nn.Dropout(.15),nn.Linear(256,2))
    def encode(self, x, lengths):
        mask = torch.arange(x.size(1),device=x.device)[None]<lengths[:,None]
        memory = self.lm.encode(x,mask)
        query = torch.full((x.size(0),1),self.r['bos'],dtype=torch.long,device=x.device)
        decoded,_ = self.lm.decode(query,memory,mask)
        mean = (memory.float()*mask[:,:,None]).sum(1)/lengths[:,None]
        return torch.cat((mean,decoded[:,0].float()),-1)
    def forward(self, tokens):
        starts = chunk_starts(len(tokens),self.r,self.coverage)
        device = self.lm.embed.weight.device
        reps = []
        for at in range(0,len(starts),self.r['chunk_batch']):
            group = starts[at:at+self.r['chunk_batch']]
            parts = [np.append(tokens[s:s+self.r['length']-1],self.r['eos']) for s in group]
            lengths = torch.tensor([len(v) for v in parts],device=device)
            width = math.ceil(max(map(len,parts))/8)*8
            x = torch.full((len(parts),width),self.r['pad'],dtype=torch.long,device=device)
            for i,v in enumerate(parts): x[i,:len(v)] = torch.as_tensor(v,dtype=torch.long,device=device)
            h = checkpoint(self.encode,x,lengths,use_reentrant=False) if self.training else self.encode(x,lengths)
            reps.append(h)
        pos = torch.tensor(starts,dtype=torch.float32,device=device)[:,None]/max(1,len(tokens)-1)
        h = self.chunk_head(torch.cat((torch.cat(reps),pos),-1))
        weights = self.pool(h).float().softmax(0)
        return self.head((h*weights).sum(0,keepdim=True))

def chunk_starts(size, r, coverage):
    last = max(0,size-(r['length']-1))
    starts = list(range(0,last+1,r['stride']))
    if starts[-1]!=last: starts.append(last)
    if coverage=='head_tail': return sorted(set([0,last]))
    if coverage=='uniform' and len(starts)>r['chunks']:
        starts = [starts[i] for i in np.linspace(0,len(starts)-1,r['chunks'],dtype=int)]
    return starts

def make_model(method, c, r, methods):
    saved = torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
    config = saved.get('config',c).copy()
    config['gradient_checkpointing'] = True
    config['pad_id'], config['bos_id'], config['eos_id'] = r['pad'],r['bos'],r['eos']
    state = saved['model']
    if method=='distill':
        smaller = {k:v for k,v in state.items() if not k.startswith(('encoder.','decoder.'))}
        for side in ['encoder','decoder']:
            chosen = np.linspace(0,config[side+'_layers']-1,4,dtype=int).tolist()
            for i,j in enumerate(chosen):
                smaller.update({f'{side}.{i}.'+k.split('.',2)[2]:v for k,v in state.items()
                                if k.startswith(f'{side}.{j}.')})
            config[side+'_layers'] = len(chosen)
        state = smaller
    lm = LegalSeq2Seq(config)
    lm.load_state_dict(state,strict=True)
    del saved,state
    if method not in ['full','distill']: lm.requires_grad_(False)
    if method=='partial':
        for b in list(lm.encoder[-2:])+list(lm.decoder[-3:]): b.requires_grad_(True)
        lm.encoder_norm.requires_grad_(True)
        lm.decoder_norm.requires_grad_(True)
    targets = ['qkv','q','kv','out','gate','up','down']
    if method in ['lora','lorma','loha']:
        for name,layer in list(lm.named_modules()):
            if not isinstance(layer,nn.Linear): continue
            key = name.split('.')[-1]
            wrapped = None
            if method=='lora' and key in targets: wrapped = LoRALinear(layer,r['rank'],2*r['rank'])
            if method=='lorma' and key in targets: wrapped = LoRMALinear(layer,r['rank'],2*r['rank'])
            if method=='loha' and key in ['qkv','q','kv','out']: wrapped = LoHaLinear(layer)
            if wrapped is not None:
                parent,key = name.rsplit('.',1)
                setattr(lm.get_submodule(parent),key,wrapped)
    model = CaseModel(lm,method,r,methods[method]['coverage'])
    return model

def case_tokens(row, cache):
    return cache[row['offset']:row['offset']+row['size']]

@torch.inference_mode()
def predict(model, rows, cache):
    model.eval()
    logits = np.empty((len(rows),2),dtype=np.float32)
    for i,row in enumerate(rows):
        with torch.autocast('cuda',dtype=torch.float16):
            z = model(case_tokens(row,cache))
        logits[i] = z.float().cpu().numpy()[0]
    return logits

def probabilities(z):
    z = z-z.max(axis=1,keepdims=True)
    p = np.exp(z)
    return p/p.sum(axis=1,keepdims=True)

def choose_threshold(gold, p):
    thresholds = sorted(np.linspace(.2,.8,61),key=lambda t:abs(t-.5))
    scores = [f1_score(gold,p>=t,labels=[0,1],average='macro',zero_division=0) for t in thresholds]
    k = int(np.argmax(scores))
    return float(thresholds[k]),float(scores[k])

def write_submission(rows, p, threshold, path):
    if len(p)!=len(rows) or not np.isfinite(p).all():
        raise ValueError('Invalid case predictions')
    payload = json.dumps([dict(id=row['id'],pred=int(v>=threshold)) for row,v in zip(rows,p)],
                         ensure_ascii=False,indent=2)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z: z.writestr('prediction.json',payload)
    return hashlib.sha256(payload.encode()).hexdigest()


## Training, validation and compact checkpoints

In [37]:
def run_method(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    folder.mkdir(exist_ok=True)
    if (folder/'done.json').exists():
        print(f'GPU {gpu} | {method} already complete; reusing outputs', flush=True)
        return
    with (root/'train_dev.pkl').open('rb') as f: data = cloudpickle.load(f)
    train,dev = data['train'],data['dev']
    caches = {s:np.memmap(root/f'{s}_tokens.bin',dtype=np.int32,mode='r') for s in ['train','dev']}
    gold = np.array([v['y'] for v in dev])
    soft = None
    if method=='distill':
        tm = make_model(teacher,c,r,methods)
        ck = torch.load(root/teacher/'best.pt',map_location='cpu',weights_only=False)
        tm.load_state_dict(ck['weights'],strict=False)
        tm.to(device)
        soft = predict(tm,train,caches['train'])
        np.save(folder/'teacher_logits.npy',soft)
        del tm,ck
        gc.collect()
        torch.cuda.empty_cache()
    model = make_model(method,c,r,methods).to(device)
    params = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
    body = [p for n,p in params if n.startswith('lm.')]
    head = [p for n,p in params if not n.startswith('lm.')]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body,lr=lr,initial_lr=lr),
                             dict(params=head,lr=3e-4,initial_lr=3e-4)],weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024)
    count = np.bincount([v['y'] for v in train],minlength=r['classes'])
    weight = 1/np.sqrt(np.maximum(count,1))
    weight = np.clip(weight/weight.mean(),.25,4.)
    weight = torch.tensor(weight,dtype=torch.float32,device=device)
    nsteps = math.ceil(len(train)/(r['micro']*r['accum']))
    total = methods[method]['epochs']*nsteps
    warmup = max(10,int(.06*total))
    best,stale,step,history = -1.,0,0,[]
    start = time.monotonic()
    total_params = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for _,p in params)
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} ({torch.cuda.get_device_name(device)}) | model on {next(model.parameters()).device} | {total_params:,} parameters | {trained:,} trainable',flush=True)
    stop = False
    for epoch in range(1,methods[method]['epochs']+1):
        order = np.random.default_rng(r['seed']+epoch).permutation(len(train))
        loss_sum,seen = 0.,0
        for at in range(0,len(order),r['micro']*r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at+r['micro']*r['accum']]
            factor = min(1.,(step+1)/warmup)*(.1+.9*.5*(1+math.cos(math.pi*min(1.,step/total))))
            for g in opt.param_groups: g['lr'] = g['initial_lr']*factor
            for offset in range(0,len(group),r['micro']):
                ix = group[offset:offset+r['micro']]
                batch = [train[int(i)] for i in ix]
                y = torch.tensor([row['y'] for row in batch],device=device)
                with torch.autocast('cuda',dtype=torch.float16):
                    z = torch.cat([model(case_tokens(row,caches['train'])) for row in batch])
                    ce = (F.cross_entropy(z.float(),y,reduction='none')*weight[y]).mean()
                    loss = ce
                    if soft is not None:
                        target = torch.from_numpy(soft[ix]).to(device)
                        kd = F.kl_div(F.log_softmax(z.float()/2.,-1),
                                      F.softmax(target/2.,-1),reduction='batchmean')*4.
                        loss = .5*ce+.5*kd
                    scaled = loss*(len(ix)/len(group))
                scaler.scale(scaled).backward()
                loss_sum += loss.detach().item()*len(ix)
                seen += len(ix)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _,p in params],1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            evaluate = step%r['eval_steps']==0 or at+len(group)>=len(order)
            if evaluate:
                logits = predict(model,dev,caches['dev'])
                score = f1_score(gold,logits.argmax(1),labels=list(range(r['classes'])),
                                 average='macro',zero_division=0)
                improved = score>best
                stale = 0 if improved else stale+1
                if improved:
                    best = float(score)
                    weights = {n:p.detach().cpu().float() for n,p in params}
                    torch.save(dict(weights=weights,method=method,config=model.lm.cfg,
                                    epoch=epoch,step=step,score=best,teacher=teacher),folder/'best.tmp')
                    os.replace(folder/'best.tmp',folder/'best.pt')
                    np.save(folder/'dev_logits.npy',logits)
                row = dict(epoch=epoch,step=step,loss=loss_sum/max(seen,1),macro_f1=float(score))
                history.append(row)
                (folder/'history.json').write_text(json.dumps(history,indent=2))
                print(f'{method} | epoch {epoch} step {step} | loss {row["loss"]:.4f} | dev F1 {score:.4f}',flush=True)
                if stale>=r['patience']:
                    stop=True
                    break
        if stop: break
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    dev_logits = np.load(folder/'dev_logits.npy')
    threshold,calibrated = choose_threshold(gold,probabilities(dev_logits)[:,1])
    pred = (probabilities(dev_logits)[:,1]>=threshold).astype(int)
    report = classification_report(gold,pred,labels=list(range(r['classes'])),
                                   target_names=['0','1'],zero_division=0,output_dict=True)
    (folder/'classification_report.json').write_text(json.dumps(report,indent=2))
    result = dict(method=method,run_id=r['run_id'],macro_f1=best,parameters=total_params,trainable=trained,
                  epoch=ck['epoch'],step=ck['step'],last_step=step,teacher=teacher,
                  stopping_reason='patience' if stop else 'epoch_limit',
                  minutes=(time.monotonic()-start)/60,
                  peak_gb=torch.cuda.max_memory_allocated(device)/2**30)
    result.update(threshold=threshold, calibrated_macro_f1=calibrated,
                  coverage=methods[method]['coverage'])
    (folder/'done.json').write_text(json.dumps(result,indent=2))


## Two GPUs

In [38]:
def run_jobs(names, worker=run_method):
    pending, running, finished = (list(names), {}, set())
    first = ['full', 'partial', 'lora', 'lorma', 'loha']
    while pending or running:
        for gpu, (process, path, method) in list(running.items()):
            status = process.poll()
            if status is not None:
                if status != 0:
                    raise subprocess.CalledProcessError(status, process.args)
                path.unlink()
                del running[gpu]
                finished.add(method)
                print(f'GPU {gpu} finished {method}', flush=True)
        for gpu in range(2):
            if gpu in running:
                continue
            ready = [m for m in pending if worker is not run_method or m != 'distill' or set(first).issubset(finished)]
            if not ready:
                continue
            method = ready[0]
            teacher = None
            if worker is run_method and method == 'distill':
                teacher = max(first, key=lambda m: json.loads((OUT / m / 'done.json').read_text())['macro_f1'])
            path = OUT / f'worker_{gpu}.pkl'
            with path.open('wb') as f:
                cloudpickle.dump((worker, (gpu, method, C, R, METHODS, teacher)), f)
            command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
            process = subprocess.Popen([sys.executable, '-u', '-c', command, str(path)], env=os.environ.copy())
            running[gpu] = (process, path, method)
            pending.remove(method)
            print(f'GPU {gpu} starting {method} | PID {process.pid}', flush=True)
        if pending and (not running):
            raise RuntimeError('Distillation requires the first five methods in this job queue')
        if running:
            time.sleep(0.5)
run_jobs(list(METHODS))


GPU 0 starting full | PID 470
GPU 1 starting partial | PID 471
GPU 0 | full already complete; reusing outputs
GPU 1 | partial already complete; reusing outputs
GPU 1 finished partial
GPU 1 starting lora | PID 488
GPU 0 finished full
GPU 0 starting lorma | PID 490
GPU 1 | lora already complete; reusing outputs
GPU 0 | lorma already complete; reusing outputs
GPU 1 finished lora
GPU 1 starting loha | PID 506
GPU 0 finished lorma
GPU 1 | loha already complete; reusing outputs
GPU 1 finished loha
GPU 0 starting distill | PID 515
distill | GPU 0 (Tesla T4) | model on cuda:0 | 91,047,683 parameters | 91,047,683 trainable
distill | epoch 1 step 200 | loss 0.4159 | dev F1 0.3333
distill | epoch 1 step 400 | loss 0.4199 | dev F1 0.4208
distill | epoch 1 step 600 | loss 0.4168 | dev F1 0.4574
distill | epoch 1 step 800 | loss 0.4078 | dev F1 0.4957
distill | epoch 1 step 1000 | loss 0.3934 | dev F1 0.5380
distill | epoch 1 step 1200 | loss 0.3837 | dev F1 0.4793
distill | epoch 1 step 1400 | loss

## Test inference after training

In [39]:
path = hf_hub_download(REPO, 'Phase_4_CJPE/Test/test.json', repo_type='dataset',
                       revision=revision, token=HF_TOKEN)
docs = read_docs(json.loads(Path(path).read_text()), labelled=False)
DATA['test'] = cache_docs('test',docs)
with (OUT/'test.pkl').open('wb') as f: cloudpickle.dump(DATA['test'],f)
del docs

def run_inference(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    with (root/'test.pkl').open('rb') as f: test = cloudpickle.load(f)
    cache = np.memmap(root/'test_tokens.bin',dtype=np.int32,mode='r')
    model = make_model(method,c,r,methods)
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(ck['weights'],strict=False)
    del ck
    model.to(device)
    torch.cuda.synchronize(device)
    start = time.monotonic()
    logits = predict(model,test,cache)
    torch.cuda.synchronize(device)
    seconds = time.monotonic()-start
    np.save(folder/'test_logits.npy',logits)
    result = json.loads((folder/'done.json').read_text())
    sub = root/'submissions'
    sub.mkdir(exist_ok=True)
    write_submission(test,probabilities(logits)[:,1],result['threshold'],
                     sub/f'submission_P4_A2_{method}.zip')
    result.update(test_seconds=seconds,test_cases_per_second=len(test)/max(seconds,1e-6))
    (folder/'done.json').write_text(json.dumps(result,indent=2))
    print(f'{method} | saved test submission',flush=True)

run_jobs(list(METHODS),worker=run_inference)


test: 1,517 cases | 7,856,191 BPE tokens
GPU 0 starting full | PID 534
GPU 1 starting partial | PID 535
partial | saved test submission
GPU 1 finished partial
GPU 1 starting lora | PID 546
full | saved test submission
GPU 0 finished full
GPU 0 starting lorma | PID 552
lora | saved test submission
GPU 1 finished lora
GPU 1 starting loha | PID 558
lorma | saved test submission
GPU 0 finished lorma
GPU 0 starting distill | PID 564
distill | saved test submission
GPU 0 finished distill
loha | saved test submission
GPU 1 finished loha


In [32]:
for method in ['full', 'partial', 'lora', 'loha', 'lorma']:
    result = json.loads((OUT/method/'done.json').read_text())
    print(method, result['run_id'])

full 5676240ecf05e01ba6d508a7cca5d5e3cd0d9e8c6cf0a58ebe96651f067af84b
partial 5676240ecf05e01ba6d508a7cca5d5e3cd0d9e8c6cf0a58ebe96651f067af84b
lora 5676240ecf05e01ba6d508a7cca5d5e3cd0d9e8c6cf0a58ebe96651f067af84b
loha 5676240ecf05e01ba6d508a7cca5d5e3cd0d9e8c6cf0a58ebe96651f067af84b
lorma 5676240ecf05e01ba6d508a7cca5d5e3cd0d9e8c6cf0a58ebe96651f067af84b


## Seven submissions

In [40]:
gold = np.array([v['y'] for v in DATA['dev']])
results = [json.loads((OUT/m/'done.json').read_text()) for m in METHODS]
ranked = sorted(results,key=lambda v:v['calibrated_macro_f1'],reverse=True)
members = [v['method'] for v in ranked[:3]]
dev_probs = np.stack([probabilities(np.load(OUT/m/'dev_logits.npy'))[:,1] for m in members])
test_probs = np.stack([probabilities(np.load(OUT/m/'test_logits.npy'))[:,1] for m in members])
weights = np.array([max(v['calibrated_macro_f1'],1e-6) for v in ranked[:3]])
candidates = [np.ones(3)/3, weights/weights.sum()]
choices = [choose_threshold(gold,w@dev_probs) for w in candidates]
k = int(np.argmax([score for threshold,score in choices]))
w, (threshold,score) = candidates[k],choices[k]
ensemble = w@test_probs
selection = dict(members=members,weights=w.tolist(),threshold=threshold,dev_macro_f1=score)
(OUT/'ensemble.json').write_text(json.dumps(selection,indent=2))
SUB = OUT/'submissions'
SUB.mkdir(exist_ok=True)
fingerprints = {}
for row in results+[dict(method='ensemble',threshold=threshold)]:
    method = row['method']
    p = ensemble if method=='ensemble' else probabilities(np.load(OUT/method/'test_logits.npy'))[:,1]
    digest = write_submission(DATA['test'],p,row['threshold'],SUB/f'submission_P4_A2_{method}.zip')
    if digest in fingerprints: print(f'Identical predictions: {method} and {fingerprints[digest]}')
    fingerprints[digest] = method
results.append(dict(method='ensemble',calibrated_macro_f1=score,**selection))
(OUT/'results.json').write_text(json.dumps(results,indent=2))
for v in sorted(results,key=lambda v:v['calibrated_macro_f1'],reverse=True):
    print(f'{v["method"]:10s} dev Macro-F1 {v["calibrated_macro_f1"]:.4f}')
print(f'Seven submission ZIPs: {SUB}')


ensemble   dev Macro-F1 0.6213
full       dev Macro-F1 0.6209
distill    dev Macro-F1 0.6124
loha       dev Macro-F1 0.5591
lora       dev Macro-F1 0.5553
lorma      dev Macro-F1 0.5486
partial    dev Macro-F1 0.5440
Seven submission ZIPs: /kaggle/working/phase4_a4/submissions


In [41]:
with zipfile.ZipFile(OUT/'all_submissions_P4_A2.zip','w',zipfile.ZIP_DEFLATED) as z:
    for method in list(METHODS)+['ensemble']:
        p = SUB/f'submission_P4_A2_{method}.zip'
        z.write(p,p.name)
    for name in ['results.json','ensemble.json','manifest.json']: z.write(OUT/name,name)
print(OUT/'all_submissions_P4_A2.zip')


/kaggle/working/phase4_a4/all_submissions_P4_A2.zip


## Outputs

## Adapter implementations